# **Pre-Labeling of Argentine ID Card elements**

## **Objective**
The main objective of this notebook is build an automatic *pre-labeling* pipeline for the regions of interest (ROIs) of the front and back of the Argentine ID Card, reusing the relatives coords defined in `00-argentinian-ID-card-detection/config/coords_data.json`.

As the result of the extraction and geometric correction process in the previous notebook ([ 00_extract_DNI_from_image_v1.0.0.ipynb](./00_extract_DNI_from_image_v1.0.0.ipynb)), the Argentine ID Cards don't have backgrounds and preserv a fixed aspect ratio ($1200 \times 756\text{ px}$), this ensures that the location of the internal elements remains constant in all samples.

## **Expected Input**
* **Corrected Images `front`:** `data/prc/DNIs/processed/front/`.
* **Corrected Images `back`:** `data/prc/DNIs/processed/back/`.
* **Coordinates JSON file:** `research/00-argentinian-ID-card-detection/config/coords_data.json`.

## **Expected Output**
* **Labeling Directory:** `data/prc/DNIs/processed/labels/`.
* **YOLO Anotation file:** `.txt` files for each sample with the next format `<class_id> <x_center> <y_center> <width> <height>`.
* **Pre-Labeling Dataset:** First version of our *dataset*, ready to be load and verify in **FiftyOne** and **CVAT** before the process of fine-tuning for the task of detect elements from ID-Card.

## **Example**
<table>
  <tr>
    <td align="center" style="padding: 10px;">
      <img src="./assets/00_3.png" width="420"><br>
      <em>Figure 1. Input image.</em>
    </td>
    <td align="center" style="padding: 10px;">
      <img src="./assets/00_4.png" width="420"><br>
      <em>Figura 2. Annotated ID-Card elements.</em>
    </td>
  </tr>
</table>

## **Libraries and Settings**

In [16]:
from tqdm import tqdm
from pathlib import Path
import os
import json

if not os.environ.get("BASE_PROJECT_DIR"):
    os.environ["BASE_PROJECT_DIR"] = input("Enter the base project directory path: ")
INPUT_FRONT_IMAGES = Path(os.environ["BASE_PROJECT_DIR"]) / "data/prc/DNIs/processed/front"
INPUT_BACK_IMAGES = Path(os.environ["BASE_PROJECT_DIR"]) / "data/prc/DNIs/processed/back"
OUTPUT_LABELS_PATH = Path(os.environ["BASE_PROJECT_DIR"]) / "data/prc/DNIs/processed/labels"
COORDS_DATA_PATH = Path(os.environ["BASE_PROJECT_DIR"]) / "research/00-argentinian-ID-card-detection/config/coords_data.json"
CLASSES = {
    "picture": 0,
    "shield": 1,
    "doc_number": 2,
    "tramite_number": 3,
    "pdf_417": 4,
    "mrz": 5,
    "address": 6,
    "gender": 7,
    "country": 8,
}

* Load our relatives coordinates json defined in the previous experiment.

In [17]:
with open(COORDS_DATA_PATH, "r") as f:
    coords_data = json.load(f)

* Since the coordinates of each element represent the top left and bottom right points of the bounding box, we need to transform it into a format valid for **YOLO** (`<class_id> <x_center> <y_center> <width> <height>`).
* To resolve this problem define two functions
  * `convert_to_yolo_format()` to transform our coordinates to a valid form for **YOLO**.
  * `annotate()` to annotate an image with a bounding box and label.
* Define an auxiliar functions to get a value from `dict` through a `key`.

In [3]:
def convert_to_yolo_format(coords: list) -> str:
    """Convert bounding box coordinates to YOLO format"""
    pt1, pt2 = coords
    width = pt2[0] - pt1[0]
    height = pt2[1] - pt1[1]

    x_center = pt1[0] + width / 2
    y_center = pt1[1] + height / 2

    return f"{x_center} {y_center} {width} {height}"

def annotate(image_path: Path,
             coords: list,
             label: str,
             label_path: Path = None) -> None:
    """Annotate an image with a bounding box and label"""
    out = label_path / f"{image_path.stem}.txt"
    with open(out, "a") as f:
        f.write(f"{label} {convert_to_yolo_format(coords)}\n")

def get_value_by_key(data: dict, key: str):
    """Get value from a dictionary by key, return None if key not found"""
    return data.get(key, None)

In [4]:
def execute(input_dir: Path,
            type_ID: str,
            coords: dict,
            coords_key: list[str],
            classes: dict,
            classes_key: list[str],
            output_dir: Path) -> None:
    """Execute the annotation process for a given input directory and coordinates"""
    images = list(Path(input_dir / type_ID).glob("*"))
    os.makedirs(output_dir, exist_ok=True)
    print(f"Found {len(images)} {type_ID} images of Argentine ID Card.")
    print(f"Will be generated {len(images)} annotation files for {type_ID} images with YOLO format.")

    for image in tqdm(images, desc=f"Annotating {type_ID} images"):
        for coord_key, class_key in zip(coords_key, classes_key):
            annotate(image_path=image,
                     coords=get_value_by_key(coords, coord_key),
                     label=get_value_by_key(classes, class_key),
                     label_path=output_dir)


## **Annotate the front elements of DNI type ID-Card**

* Extract the coordinates for the front of the *type* `dni`.
* Execute the pipeline.

In [18]:
dni_coords = coords_data["dni"]
front_coords = dni_coords["front"]
front_coords

{'PICTURE_COORD': [[0.04844290657439446, 0.2717391304347826],
  [0.3460207612456747, 0.8152173913043478]],
 'SHIELD_COORD': [[0.04152249134948097, 0.09782608695652174],
  [0.11418685121107267, 0.2554347826086957]],
 'FIRST_NAME_COORD': [[0.370242214532872, 0.40217391304347827],
  [0.6920415224913494, 0.46195652173913043]],
 'LAST_NAME_COORD': [[0.370242214532872, 0.2717391304347826],
  [0.6920415224913494, 0.33152173913043476]],
 'DOCUMENT_NUMBER_COORD': [[0.06920415224913495, 0.8858695652173914],
  [0.31141868512110726, 0.9782608695652174]],
 'SEX_COORD': [[0.370242214532872, 0.5163043478260869],
  [0.44982698961937717, 0.5652173913043478]],
 'NATIONALITY_COORD': [[0.49480968858131485, 0.5163043478260869],
  [0.726643598615917, 0.5652173913043478]],
 'BIRTH_DATE_COORD': [[0.370242214532872, 0.6086956521739131],
  [0.6920415224913494, 0.657608695652174]],
 'NRO_TRAMITE_COORD': [[0.370242214532872, 0.8804347826086957],
  [0.5536332179930796, 0.9239130434782609]],
 'OFFICE_CODE_COORD': [

In [19]:
execute(
    input_dir=INPUT_FRONT_IMAGES,
    type_ID="dni",
    coords=front_coords,
    coords_key=["PICTURE_COORD", "SHIELD_COORD", "DOCUMENT_NUMBER_COORD", "SEX_COORD", "NRO_TRAMITE_COORD"],
    classes=CLASSES,
    classes_key=["picture", "shield", "doc_number", "gender", "nro_tramite"],
    output_dir=OUTPUT_LABELS_PATH
)

Found 128 dni images of Argentine ID Card.
Will be generated 128 annotation files for dni images with YOLO format.


Annotating dni images: 100%|██████████| 128/128 [00:00<00:00, 6933.81it/s]


## **Annotate the back elements of DNI type ID-Card**

* Extract the coordinates for the back of the *type* `dni`.
* Execute the pipeline.

In [20]:
dni_coords = coords_data["dni"]
back_coords = dni_coords["back"]
back_coords

{'ADDRESS_COORD': [[0.15918958031837915, 0.034403669724770644],
  [0.8683068017366136, 0.0871559633027523]],
 'MRZ_COORD': [[0.05788712011577424, 0.6422018348623854],
  [0.9696092619392185, 0.963302752293578]],
 'COUNTRY_COORD': [[0.6512301013024602, 0.2981651376146789],
  [0.7525325615050651, 0.573394495412844]]}

In [21]:
execute(
    input_dir=INPUT_BACK_IMAGES,
    type_ID="dni",
    coords=back_coords,
    coords_key=["ADDRESS_COORD", "MRZ_COORD", "COUNTRY_COORD"],
    classes=CLASSES,
    classes_key=["address", "mrz", "country"],
    output_dir=OUTPUT_LABELS_PATH
)

Found 82 dni images of Argentine ID Card.
Will be generated 82 annotation files for dni images with YOLO format.


Annotating dni images: 100%|██████████| 82/82 [00:00<00:00, 8708.49it/s]


## **Annotate the front elements of Card type ID-Card**

* Extract the coordinates for the front of the *type* `card`.
* Execute the pipeline.

In [22]:
card_coords = coords_data["card"]
front_coords = card_coords["front"]
front_coords

{'PICTURE_COORD': [[0.5571030640668524, 0.07570977917981073],
  [0.8555511341026661, 0.6309148264984227]],
 'SHIELD_COORD': [[0.043772383605252686, 0.056782334384858045],
  [0.1054516514126542, 0.20189274447949526]],
 'FIRST_NAME_COORD': [[0.043772383605252686, 0.23659305993690852],
  [0.3979307600477517, 0.2744479495268139]],
 'LAST_NAME_COORD': [[0.043772383605252686, 0.3186119873817035],
  [0.3979307600477517, 0.35962145110410093]],
 'DOCUMENT_NUMBER_COORD': [[0.043772383605252686, 0.39936908517350156],
  [0.31834460803820136, 0.4416403785488959]],
 'SEX_COORD': [[0.38599283724631916, 0.39936908517350156],
  [0.43772383605252685, 0.4416403785488959]],
 'MRZ_COORD': [[0.03979307600477517, 0.7097791798107256],
  [0.8754476721050537, 0.9337539432176656]]}

In [23]:
execute(
    input_dir=INPUT_FRONT_IMAGES,
    type_ID="card",
    coords=front_coords,
    coords_key=["PICTURE_COORD", "SHIELD_COORD", "DOCUMENT_NUMBER_COORD", "SEX_COORD", "MRZ_COORD"],
    classes=CLASSES,
    classes_key=["picture", "shield", "doc_number", "gender", "mrz"],
    output_dir=OUTPUT_LABELS_PATH
)

Found 60 card images of Argentine ID Card.
Will be generated 60 annotation files for card images with YOLO format.


Annotating card images: 100%|██████████| 60/60 [00:00<00:00, 4926.46it/s]


## **Annotate the back elements of Card type ID-Card**

* Extract the coordinates for the back of the *type* `card`.
* Execute the pipeline.

In [24]:
card_coords = coords_data["card"]
back_coords = card_coords["back"]
back_coords

{'ADDRESS_COORD': [[0.16997792494481237, 0.08916323731138547],
  [0.6181015452538632, 0.12345679012345678]],
 'NRO_TRAMITE_COORD': [[0.06622516556291391, 0.6742112482853223],
  [0.25165562913907286, 0.7133058984910837]],
 'COUNTRY_COORD': [[0.3598233995584989, 0.6310013717421125],
  [0.45474613686534215, 0.8916323731138546]]}

In [25]:
execute(
    input_dir=INPUT_BACK_IMAGES,
    type_ID="card",
    coords=back_coords,
    coords_key=["ADDRESS_COORD", "NRO_TRAMITE_COORD", "COUNTRY_COORD"],
    classes=CLASSES,
    classes_key=["address", "nro_tramite", "country"],
    output_dir=OUTPUT_LABELS_PATH
)

Found 52 card images of Argentine ID Card.
Will be generated 52 annotation files for card images with YOLO format.


Annotating card images: 100%|██████████| 52/52 [00:00<00:00, 9426.62it/s]


In [27]:
print(f"Through this process, a total of {len(list(Path(OUTPUT_LABELS_PATH).glob('*.txt')))} annotation files have been generated in YOLO format for both DNI and card images.")

Through this process, a total of 322 annotation files have been generated in YOLO format for both DNI and card images.
